# PatchDistill Colab Pilot

Run the clone/setup cells first. They copy the whole PatchDistill repository into the Colab runtime, then install dependencies from the repository root.

If `!pwd` is `/content` and `!ls` only shows `sample_data`, the repository is not on the Colab runtime yet. Opening this notebook from Cursor does not automatically copy `/home/blackleg/ws/research/llm/new/patchDistill` to Colab.

The next code cell uses the GitHub route:

```bash
git clone https://github.com/black-leg-nameko/patchDistill.git /content/patchDistill
cd /content/patchDistill
```

```python
# Route B: put the folder on Google Drive, then mount Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/patchDistill
```

After that, the install cell below will find `requirements.txt` and `patchdistill/`.

In [1]:
!nvidia-smi

Sun Jun  7 18:14:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/black-leg-nameko/patchDistill.git"
REPO_DIR = Path("/content/patchDistill")

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}; pulling latest changes.")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Using repo root:", Path.cwd())
print("Top-level files:")
print("\n".join(sorted(p.name for p in Path.cwd().iterdir())[:40]))

In [ ]:
MODEL_NAME = "gpt2"
RUN_NAME = "gpt2_a100_pilot"
LAYERS = "0,6,11"
N_DATA = 160
MAX_FEATURE_EXAMPLES = 80
MAX_PATCH_EXAMPLES = 12
MAX_PATCH_POSITIONS = 3
DTYPE = "bfloat16"
ATTN_IMPL = None

# For a stronger A100 pilot, try:
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# RUN_NAME = "qwen25_05b_a100_pilot"
# LAYERS = "0,12,23"
# ATTN_IMPL = "eager"

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root():
    candidates = [Path.cwd(), Path('/content/patchDistill')]
    drive = Path('/content/drive/MyDrive')
    if drive.exists():
        candidates.extend([
            drive / 'patchDistill',
            drive / 'Colab Notebooks' / 'patchDistill',
        ])
    for candidate in candidates:
        if (candidate / 'requirements.txt').exists() and (candidate / 'patchdistill').is_dir():
            return candidate
    for root in [Path('/content'), drive]:
        if root.exists():
            for req in root.rglob('requirements.txt'):
                candidate = req.parent
                if (candidate / 'patchdistill').is_dir():
                    return candidate
    return None

repo_root = find_repo_root()
if repo_root is None:
    raise FileNotFoundError(
        'PatchDistill repo root was not found. Current Colab runtime does not contain the project files. '
        'If !pwd is /content and !ls only shows sample_data, clone/upload the whole repository first. '
        'Expected files: requirements.txt and patchdistill/.'
    )

os.chdir(repo_root)
print('Using repo root:', Path.cwd())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

In [ ]:
!python -m patchdistill.cli make-data --n {N_DATA} --out data/synthetic_direct_pi.jsonl
!python -m patchdistill.cli run-surrogate --data data/synthetic_direct_pi.jsonl --out runs/surrogate_mvp --split template

In [ ]:
attn_arg = "" if ATTN_IMPL is None else f"--attn-implementation {ATTN_IMPL}"
!python -m patchdistill.cli hf-extract \
  --model {MODEL_NAME} \
  --data data/synthetic_direct_pi.jsonl \
  --out runs/{RUN_NAME}_features.jsonl \
  --max-examples {MAX_FEATURE_EXAMPLES} \
  --layers {LAYERS} \
  --dtype {DTYPE} \
  {attn_arg}

In [ ]:
!python -m patchdistill.cli hf-patch \
  --model {MODEL_NAME} \
  --data data/synthetic_direct_pi.jsonl \
  --out runs/{RUN_NAME}_patch.jsonl \
  --layers {LAYERS} \
  --max-examples {MAX_PATCH_EXAMPLES} \
  --max-positions {MAX_PATCH_POSITIONS} \
  --dtype {DTYPE} \
  {attn_arg}

In [ ]:
!python -m patchdistill.cli fit-proxy \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_proxy

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --out runs/{RUN_NAME}_detector_features_only

!python -m patchdistill.cli fit-detector \
  --features runs/{RUN_NAME}_features.jsonl \
  --patch runs/{RUN_NAME}_patch.jsonl \
  --out runs/{RUN_NAME}_detector_distilled

In [ ]:
!python -m patchdistill.cli collect-results --runs runs --out runs/summary.json --markdown runs/summary.md

from pathlib import Path
print(Path("runs/summary.md").read_text())